<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [1]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [2]:
# Parte 1: Limpieza y preparación

# 1. Revisar y describir el dataset
print("=" * 60)
print("DIMENSIONES DEL DATASET")
print("=" * 60)
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

print("\n" + "=" * 60)
print("TIPOS DE DATOS")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("PORCENTAJE DE NULOS POR COLUMNA")
print("=" * 60)
print((df.isnull().sum() / len(df) * 100).round(2))

# 2. Transformar date_added a tipo fecha
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')

# 3. Crear columnas auxiliares con assign
df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)

print("\n" + "=" * 60)
print("DATAFRAME ACTUALIZADO (primeras filas)")
print("=" * 60)
df[['title', 'date_added', 'year_added', 'month_added']].head()

DIMENSIONES DEL DATASET
Número de filas: 8807
Número de columnas: 12

TIPOS DE DATOS
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

VALORES NULOS POR COLUMNA
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

PORCENTAJE DE NULOS POR COLUMNA
show_id          0.00
type             0.00
title            0.00
director        29.91
cast             9.37
country          9.44
date_added       0.11
release_year     0.00
rating           0.05
duration         0.03
listed_in        0.00
description      0.00
dtype: float64

DATAFRAME ACTUALIZAD

,title,date_added,year_added,month_added
0,Dick Johnson Is Dead,2021-09-25,2021.0,9.0
1,Blood & Water,2021-09-24,2021.0,9.0
2,Ganglands,2021-09-24,2021.0,9.0
3,Jailbirds New Orleans,2021-09-24,2021.0,9.0
4,Kota Factory,2021-09-24,2021.0,9.0


## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [3]:
# Parte 2: Técnicas avanzadas de pandas

# 4. Películas (type == 'Movie') agregadas después de 2018
print("=" * 60)
print("4. Películas agregadas después de 2018")
print("=" * 60)
movies_after_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]
print(f"Total de películas agregadas después de 2018: {len(movies_after_2018)}")
print(movies_after_2018[['title', 'year_added']].head())

# 5. str.contains() y str.extract()
print("\n" + "=" * 60)
print("5a. Títulos que contienen 'love' (sin distinguir mayúsculas)")
print("=" * 60)
titulos_love = df.loc[df['title'].str.contains('love', case=False, na=False)]
print(f"Total: {len(titulos_love)}")
print(titulos_love[['title', 'type']].head(10))

print("\n" + "=" * 60)
print("5b. Duración en minutos para películas")
print("=" * 60)
df['duration_min'] = df.loc[df['type'] == 'Movie', 'duration'].str.extract(r'(\d+)').astype(float)
print(df[['title', 'type', 'duration', 'duration_min']].head(10))

# 6. Explode sobre listed_in
print("\n" + "=" * 60)
print("6. Explode sobre listed_in (un género por fila)")
print("=" * 60)
df_exploded = df.assign(genre=df['listed_in'].str.split(', ')).explode('genre')
print(f"Filas originales: {len(df)} | Filas tras explode: {len(df_exploded)}")
print(df_exploded[['title', 'genre']].head(10))

# 7. Top 10 géneros más frecuentes
print("\n" + "=" * 60)
print("7. Top 10 géneros más frecuentes")
print("=" * 60)
top_10_generos = df_exploded['genre'].value_counts().head(10)
print(top_10_generos)

# 8. where() y mask() para marcar películas largas
print("\n" + "=" * 60)
print("8. Películas largas (>120 min)")
print("=" * 60)
df['contenido_largo'] = df['duration_min'].mask(df['duration_min'] > 120, 'Largo').where(df['duration_min'] > 120, 'Normal')
print(df[['title', 'duration_min', 'contenido_largo']].dropna().head(10))

# 9. Películas > 100 min, rating 'R', país 'United States'
print("\n" + "=" * 60)
print("9. Películas > 100 min, rating 'R', país 'United States'")
print("=" * 60)
filtro = df.loc[
    (df['duration_min'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]
print(f"Total: {len(filtro)}")
print(filtro[['title', 'duration_min', 'rating', 'country']].head(10))

# 10. .style para top 10 películas más largas
print("\n" + "=" * 60)
print("10. Top 10 películas más largas (formateado con .style)")
print("=" * 60)
top_10_largas = df.loc[df['type'] == 'Movie'].nlargest(10, 'duration_min')[
    ['title', 'duration_min', 'country', 'rating']
]
top_10_largas.style.background_gradient(subset=['duration_min'], cmap='Reds').format({'duration_min': '{:.0f} min'})

4. Películas agregadas después de 2018
Total de películas agregadas después de 2018: 3701
                               title  year_added
0               Dick Johnson Is Dead      2021.0
6   My Little Pony: A New Generation      2021.0
7                            Sankofa      2021.0
9                       The Starling      2021.0
12                      Je Suis Karl      2021.0

5a. Títulos que contienen 'love' (sin distinguir mayúsculas)
Total: 196
                               title     type
25              Love on the Spectrum  TV Show
158          Love Don't Cost a Thing    Movie
159                   Love in a Puff    Movie
206        LSD: Love, Sex Aur Dhokha    Movie
227                      Really Love    Movie
246                      Man in Love    Movie
375                   Resort to Love    Movie
402  The Last Letter From Your Lover    Movie
485                      Lethal Love    Movie
506         This Little Love Of Mine    Movie

5b. Duración en minutos para películ

,title,duration_min,country,rating
4253,Black Mirror: Bandersnatch,312 min,United States,TV-MA
717,Headspace: Unwind Your Mind,273 min,nan,TV-G
2491,The School of Mischief,253 min,Egypt,TV-14
2487,No Longer kids,237 min,Egypt,TV-14
2484,Lock Your Girls In,233 min,nan,TV-PG
2488,Raya and Sakina,230 min,nan,TV-14
166,Once Upon a Time in America,229 min,"Italy, United States",R
7932,Sangam,228 min,India,TV-14
1019,Lagaan,224 min,"India, United Kingdom",PG
4573,Jodhaa Akbar,214 min,India,TV-14




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [4]:
# Pregunta Desafío y Bonus

# 11. Combinaciones más frecuentes de género y rating
print("=" * 60)
print("11. Combinaciones más frecuentes de género + rating")
print("=" * 60)
df_combinaciones = (
    df.assign(genre=df['listed_in'].str.split(', '))
      .explode('genre')
      .value_counts(subset=['genre', 'rating'])
      .head(15)
)
print(df_combinaciones)

# 12. Películas con mismo título pero distinto año de lanzamiento
print("\n" + "=" * 60)
print("12. Títulos con mismo nombre pero distinto release_year")
print("=" * 60)
duplicados_titulo = df[df.duplicated(subset='title', keep=False)].sort_values('title')
distintos_anio = duplicados_titulo.groupby('title').filter(lambda x: x['release_year'].nunique() > 1)
print(f"Total de títulos repetidos con distinto año: {distintos_anio['title'].nunique()}")
print(distintos_anio[['title', 'type', 'release_year']].head(10))

# 13. Títulos únicos
print("\n" + "=" * 60)
print("13. Títulos únicos")
print("=" * 60)
print(f"Total de títulos en el dataset: {len(df)}")
print(f"Total de títulos únicos: {df['title'].nunique()}")
print(f"Total de títulos duplicados: {len(df) - df['title'].nunique()}")

11. Combinaciones más frecuentes de género + rating
genre                   rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Crime TV Shows          TV-MA      350
Independent Movies      TV-MA      344
Documentaries           TV-MA      321
International Movies    TV-PG      294
Stand-Up Comedy         TV-MA      291
Name: count, dtype: int64

12. Títulos con mismo nombre pero distinto release_year
Total de títulos repetidos con distinto año: 0
Empty DataFrame
Columns: [title, type, release_year]
Index: []

13. Títulos únicos
Total de títulos en el dataset: 8807
Total de títulos únicos: 8807
Total de títulos duplicados: 0
